# Create NordForsk Awards

Creates NordForsk awards from NordForsk's own public project pages at
https://www.nordforsk.org/projects (Drupal; every project enumerated by the
site's Simple XML Sitemap). Each project page has a Facts box: Project number,
Project leader (name + institution), Call, Programme, Research area, plus a
year span and a description. **306 projects, 2013-2029** (local run 2026-09-30).

**No amounts.** NordForsk does not publish per-project amounts on its site, so
`amount` / `currency` ship NULL (Step 6.7 amount check waived).
**Dates are year-only** on the site: `start_date` = Jan 1 of the first year,
`end_date` = Dec 31 of the last year.

**Co-funding:** most NordForsk programmes are co-funded by the Nordic national
research councils (RCN, VR, Academy of Finland, IRF, DFF...). The project
number shipped here is NordForsk's own; national councils' ingests carry their
own grant numbers, so rows do not collide on id. Swedish (/sv/) pages are used
only for slugs with no English page and are de-duplicated on project number.

**Prerequisites:**
- Run `scripts/local/nordforsk_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/nordforsk/nordforsk_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the bare NordForsk Project number (5-6 digits,
e.g. `82845`, `105670`). Checked against the citation side
(`works?filter=grants.funder:F4320322885&select=grants`): citing works write the
same bare number (`82845`, `90580`, `164079`), so 162 of the 306 projects collapse
onto existing crossref/datacite/europepmc citation stubs for F4320322885.
3 project pages publish no project number and are skipped.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322885`
- display_name / ror_id / doi: from `openalex.funders.funders` (NordForsk, ror 05bqzfg94, doi 10.13039/501100004785)

**Priority:** `513` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nordforsk_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nordforsk/nordforsk_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.nordforsk_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.nordforsk_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nordforsk_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322885 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322885;

## Step 2: Create NordForsk Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nordforsk_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322885  -- NordForsk
),

people AS (
    -- Project leader(s) parsed in the script ("A and B" -> two people).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        project_number,
        TRANSFORM(
            COALESCE(from_json(people_json, 'ARRAY<STRUCT<name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>'), ARRAY()),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.nordforsk_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    -- Programme when the page names one, else the call title.
    COALESCE(NULLIF(TRIM(g.programme), ''), NULLIF(TRIM(g.call), '')) as funder_scheme,
    'nordforsk_projects' as provenance,
    TRY_TO_DATE(CONCAT(TRY_CAST(g.start_year AS INT), '-01-01'), 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(CONCAT(TRY_CAST(g.end_year AS INT), '-12-31'), 'yyyy-MM-dd') as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    TRY_CAST(g.end_year AS INT) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.nordforsk_raw g
JOIN people p ON p.project_number = g.project_number
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is a bare 4-6 digit NordForsk project number.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{4,6}$' THEN 1 ELSE 0 END) as bare_number,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^[0-9]{4,6}$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.nordforsk_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nordforsk_projects' AND priority = 513;

-- Insert into openalex_awards_raw with priority.
-- Priority 513: direct-from-funder ingest of NordForsk's own project pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 513 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    513 as priority
FROM openalex.awards.nordforsk_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_nordforsk_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.nordforsk_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.nordforsk_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.nordforsk_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.nordforsk_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.nordforsk_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Coverage (expect ~100% title/dates/description, ~99% PI, ~75% institution, 0% amount).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.nordforsk_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for NordForsk (expect ~162 ids shared; these upgrade).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(n.id) as upgraded_by_nordforsk_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.nordforsk_awards n ON n.id = c.id
WHERE c.funder_id = 4320322885 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nordforsk_projects'
GROUP BY provenance, priority;